# IMPORTAZIONE LIBRERIE UTILI

In [ ]:
import pandas as pd  
import requests 
import os 
 
from selenium import webdriver 
from selenium.webdriver.common.by import By 

# SCRAPING SITUAS

Tramite scraping scarico il dataset legato all' url di SITUAS

ATTENZIONE: Può succedere che il download del dataset ci impieghi più tempo dell' esecuzione del codice, controllare che non si blocchi la cella successiva al download, nel caso far ripartire l' esecuzione dalla suddetta cella.

In [ ]:
url_situas = "https://situas.istat.it/web/#/territorio/body?id=74&dateFrom=2020-12-31"

resp_situas = requests.get (url_situas)

print("Risposta alla richiesta:", resp_situas.status_code)

In [ ]:
#Creo il percorso di download
download_folder = os.path.abspath("fetching_data")

#Creo un oggetto che contiene tutte le impostazioni personalizzate di Chrome (al momento sarà vuoto)
chrome_options = webdriver.ChromeOptions()  

#Creo il mio dizionario di preferenze per la cartella in cui scaricarlo ed eliminazione pop up di conferma
download_pref = {"download.default_directory": download_folder,
                 "download.prompt_for_download": False } #così il download partirà in automatico

#Aggiungo alla variabile chrome option le mie preferenze, così da avere un set di opzioni
chrome_options.add_experimental_option("prefs", download_pref)

#gli dico di utilizzare i driver di chrome e quindi aprire chrome con le impostazioni che gli ho dato prima
driver = webdriver.Chrome(options=chrome_options) 

driver.get(url_situas)

In [ ]:
#Cliccare le sezioni con tempo di ritardo
import time
time.sleep(5)
driver.find_element(By.ID, 'dati-report-export-btn').click() # clicca Esporta
time.sleep(5)
driver.find_element(By.XPATH, '//*[@id="dati-report-export-btn-menu"]/li[2]/button[@title="Scarica dati in formato CSV"]').click() # clicca CSV

In [ ]:
#Creo una variabile che mi servirà per leggere il download in base alla posizione
#il csv di SITUAS avrà sempre un nome differente ad ogni download
raw_situas=os.listdir("fetching_data")
raw_situas

In [ ]:
df_situas_raw = pd.read_csv("fetching_data/" + raw_situas[0], sep=';')

df_situas_raw.head()


In [ ]:
df_situas_raw.info()

# RICHIAMO API per ISTAT 
Tramite richiesta API scarico il dataset ISTAT

ATTENZIONE: Non fare troppe richieste consecutive o potrebbero impedirtele per diverse ore

In [ ]:
url_istat = "https://esploradati.istat.it/SDMXWS/rest/data/41_983"

http_header = {'Accept': 'application/vnd.sdmx.data+csv;version=1.0.0'}

resp_istat = requests.get(url_istat, headers= http_header )

print("Risposta alla richiesta:", resp_istat.status_code)

In [ ]:
#Impongo condizione per cui se non esiste il file incidenti_istat venga creato e si scriva la risposta all' API (il dataset)
if not os.path.exists("fetching_data/incidenti_istat.csv"):
    istat_incidents = open("fetching_data/incidenti_istat.csv", "w", encoding="utf-8") #utf-8 serve per avere i caratteri corretti
    istat_incidents.write(resp_istat.text)
    istat_incidents.close()

In [ ]:
df_istat_raw= pd.read_csv("fetching_data/incidenti_istat.csv")
df_istat_raw

In [ ]:
df_istat_raw.info()